# Prévision du volume journalier des admissions aux urgences

**Approche : modèles statistiques ARIMA/SARIMAX et modèles classiques de Machine Learning.**

Ce notebook a été nettoyé pour GitHub :

- aucune donnée patient identifiable n'est affichée ;
- LSTM est entièrement supprimé ;
- Holt-Winters et ETS sont entièrement supprimés ;
- la température simulée est supprimée ;
- les variables retardées et mobiles utilisent uniquement le passé ;
- la séparation entraînement/test respecte l'ordre chronologique.


## 1. Objectif et données

L'objectif est de prévoir le nombre quotidien d'admissions aux urgences afin d'appuyer la planification des ressources. Le fichier privé brut doit être agrégé avant l'analyse. Le pipeline ne conserve que deux colonnes : `Date` et `patients`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import prepare_daily_volume
from src.features import build_feature_frame
from src.experiment import run_experiment

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "daily_er_volume.csv"
DEMO_PATH = PROJECT_ROOT / "data" / "demo" / "daily_er_volume_demo.csv"
USE_DEMO_DATA = False  # True uniquement pour une démonstration synthétique

selected_path = DEMO_PATH if USE_DEMO_DATA else DATA_PATH
if not selected_path.exists():
    raise FileNotFoundError(
        f"Fichier absent: {selected_path}\n"
        "Préparez les données avec: python prepare_data.py --input <fichier.csv>\n"
        "ou générez la démo: python scripts/generate_demo_data.py"
    )


## 2. Chargement sécurisé et agrégation

La fonction suivante accepte soit le fichier brut d'admissions, soit un fichier déjà agrégé. Elle ne retourne jamais les identifiants, noms ou autres attributs individuels.


In [ ]:
daily = prepare_daily_volume(selected_path)
print(f"Période: {daily['Date'].min().date()} → {daily['Date'].max().date()}")
print(f"Nombre de jours: {len(daily)}")
display(daily.head())

plt.figure(figsize=(14, 5))
plt.plot(daily["Date"], daily["patients"])
plt.title("Volume journalier des admissions aux urgences")
plt.xlabel("Date")
plt.ylabel("Patients")
plt.grid(True, alpha=0.3)
plt.show()


## 3. Analyse exploratoire

Cette section examine la distribution globale et l'effet du jour de la semaine. Les résultats doivent être interprétés comme des tendances descriptives, sans établir de causalité.


In [ ]:
display(daily["patients"].describe())

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekly = (
    daily.assign(weekday=daily["Date"].dt.day_name())
    .groupby("weekday")["patients"]
    .agg(["mean", "median", "std", "count"])
    .reindex(weekday_order)
)
display(weekly)

plt.figure(figsize=(10, 4))
plt.bar(weekly.index, weekly["mean"])
plt.title("Admissions moyennes par jour de la semaine")
plt.xticks(rotation=30)
plt.grid(True, axis="y", alpha=0.3)
plt.show()


## 4. Feature engineering sans fuite de données

Les variables incluent le calendrier, les retards de 1 à 28 jours, et les statistiques mobiles calculées après un décalage d'un jour. La valeur cible du jour à prédire n'est donc pas utilisée dans ses propres variables explicatives.


In [ ]:
model_df, feature_columns = build_feature_frame(daily)
print("Observations utilisables:", len(model_df))
print("Nombre de variables:", len(feature_columns))
display(model_df[["Date", "patients"] + feature_columns[:8]].head())


## 5. Modèles évalués

Le dépôt compare :

- baselines : moyenne, dernière valeur, retard saisonnier 7 jours, moyenne mobile 7 jours ;
- modèles statistiques : ARIMA et SARIMAX hebdomadaire ;
- modèles ML : régressions linéaires régularisées, KNN, SVR, Random Forest, Extra Trees, Gradient Boosting, HistGradientBoosting et MLP ;
- modèles optionnels : XGBoost, LightGBM et CatBoost.

**LSTM, Holt-Winters et ETS ne font pas partie du projet.**


In [ ]:
artifacts = run_experiment(
    data_path=selected_path,
    output_dir=PROJECT_ROOT / "results" / "notebook_run",
    figures_dir=PROJECT_ROOT / "figures" / "notebook_run",
    horizon=30,
    include_extended=True,
    include_optional=False,
)
results = artifacts["results"]
display(results.head(15))


## 6. Comparaison et diagnostic

Le choix final ne repose pas seulement sur le MAE. Il faut également examiner le RMSE, la stabilité en validation temporelle, les résidus, l'interprétabilité et l'utilité opérationnelle.


In [ ]:
best_name = artifacts["best_model"]
print("Meilleur modèle selon le MAE:", best_name)

plt.figure(figsize=(12, 7))
plot_df = results.nsmallest(15, "MAE").sort_values("MAE")
plt.barh(plot_df["model"], plot_df["MAE"])
plt.title("Comparaison des meilleurs modèles selon le MAE")
plt.xlabel("MAE")
plt.grid(True, axis="x", alpha=0.3)
plt.show()

plt.figure(figsize=(14, 5))
plt.plot(artifacts["test_dates"], artifacts["y_test"], label="Réel")
plt.plot(artifacts["test_dates"], artifacts["best_prediction"], label="Prévu")
plt.title(f"Prédictions du meilleur modèle — {best_name}")
plt.xlabel("Date")
plt.ylabel("Patients")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 7. Validation temporelle

`TimeSeriesSplit` conserve l'ordre temporel et évalue la stabilité de trois modèles d'ensemble. Cette validation est plus adaptée qu'une validation croisée aléatoire pour une série chronologique.


In [ ]:
display(artifacts["cv_summary"])


## 8. Prévision des 30 prochains jours

Lorsque le meilleur modèle est ARIMA ou SARIMAX, la prévision future est produite directement par ce modèle réentraîné sur toute la série. Lorsque le meilleur modèle est un modèle ML, une stratégie récursive utilise uniquement l'historique disponible et les prévisions antérieures.


In [ ]:
future = artifacts["future_forecast"]
display(future)

plt.figure(figsize=(14, 5))
plt.plot(daily["Date"].tail(90), daily["patients"].tail(90), label="Historique")
plt.plot(future["Date"], future["forecast_patients"], label="Prévision")
plt.title("Prévision du volume des urgences sur 30 jours")
plt.xlabel("Date")
plt.ylabel("Patients")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Résultats validés du dépôt

Une exécution complète sur le dataset fourni est documentée dans `docs/RESULTS.md`. Le meilleur modèle selon la MAE est SARIMAX(1,0,1) × (1,0,1,7), avec une MAE de 2,908 patients sur 111 jours de test. Les métriques et graphiques validés sont disponibles dans `results/final/` et `figures/final/`.


## 9. Limites et interprétation

- Les performances doivent être comparées à des baselines simples.
- Un R² faible indique que la variabilité quotidienne reste difficile à expliquer, même si le MAE est légèrement meilleur.
- Les prévisions ne constituent pas une recommandation clinique.
- Les données patient brutes ne doivent jamais être déposées dans le dépôt public.
- Les résultats de référence du README proviennent de l'exécution originale sur 9 216 admissions agrégées en 579 jours.
